# mesh_gpu — run the CUDA exercises on Google Colab (no Docker)

This notebook compiles the **existing** `source/cuda/*.cu` with `nvcc` and runs
them on Colab's free GPU. The project source is not modified — the same files
build here and inside the Docker image.

First set a GPU runtime: **Runtime -> Change runtime type -> Hardware accelerator: GPU** (the free tier gives an NVIDIA **T4** = `sm_75`).

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the project source
Clone the repo. If it is **private**, either put a token in the URL, or skip the
clone and upload the `source/` folder with the file browser on the left.

In [ ]:
REPO_URL = "https://github.com/julesdichamp/mesh_gpu.git"
!git clone --depth 1 $REPO_URL mesh_gpu
%cd mesh_gpu

## 3. Build the dependencies
libMeshb (built from source into `/content/local`) and the header-only cxxopts.
Thrust ships with the CUDA toolkit, so nothing to install for it.

In [ ]:
%%bash
set -e
git clone --depth 1 https://github.com/LoicMarechal/libMeshb /content/libMeshb
cmake -S /content/libMeshb -B /content/libMeshb/build -DCMAKE_BUILD_TYPE=Release \
      -DCMAKE_INSTALL_PREFIX=/content/local > /tmp/libmeshb_build.log 2>&1
cmake --build /content/libMeshb/build -j2 >> /tmp/libmeshb_build.log 2>&1
cmake --install /content/libMeshb/build >> /tmp/libmeshb_build.log 2>&1
echo 'libMeshb installed to /content/local'

mkdir -p /content/include
curl -sL https://raw.githubusercontent.com/jarro2783/cxxopts/v3.2.0/include/cxxopts.hpp \
     -o /content/include/cxxopts.hpp
echo 'cxxopts header ready'

## 4. Download a mesh (Zenodo)

In [ ]:
%%bash
set -e
mkdir -p data
base=https://zenodo.org/api/records/14968025/files
curl -sL $base/unit_square_132.meshb/content -o data/unit_square_132.meshb
curl -sL $base/unit_square_11k.meshb/content -o data/unit_square_11k.meshb
ls -la data

## 5. Compile the existing CUDA sources
The free GPU is a T4 (`sm_75`). If `nvidia-smi` above shows a different card,
change `-arch` (V100=70, A100=80, L4=89) or use `-arch=native`.

In [ ]:
%%bash
set -e
for f in area_triangles area_median_cells; do
  nvcc -O2 -std=c++17 -arch=sm_75 \
    -I source -I /content/include -I /content/local/include \
    source/cuda/$f.cu -L /content/local/lib -lMeshb.8 -o ${f}_cuda
  echo "built ${f}_cuda"
done

## 6. Run them
Each prints its total; the two totals should match. CSVs are written to `output/`.

In [ ]:
%%bash
./area_triangles_cuda      data/unit_square_132.meshb
./area_median_cells_cuda   data/unit_square_132.meshb --method geometric
./area_median_cells_cuda   data/unit_square_132.meshb --method third
echo '--- CSV heads ---'
head -3 output/triangle_areas.csv
head -3 output/median_cell_areas.csv